load the label file:

In [14]:
import pandas as pd

y_df = pd.read_csv('/data1/r20user3/shared_project/Hist2Cell/data/brca_clinical_from_cell_paper/filtered_subtyping_label.csv')
y_df

,Slide.ID,Case.ID,Final Pathology
0,TCGA-E2-A1IE-01Z-00-DX1,TCGA-E2-A1IE,IDC
1,TCGA-A2-A0CP-01Z-00-DX1,TCGA-A2-A0CP,IDC
2,TCGA-C8-A278-01Z-00-DX1,TCGA-C8-A278,IDC
3,TCGA-EW-A3E8-01Z-00-DX1,TCGA-EW-A3E8,ILC
4,TCGA-A2-A0YL-01Z-00-DX1,TCGA-A2-A0YL,ILC
...,...,...,...
460,TCGA-AC-A2FO-01Z-00-DX1,TCGA-AC-A2FO,ILC
461,TCGA-E9-A1R0-01Z-00-DX1,TCGA-E9-A1R0,IDC
462,TCGA-C8-A132-01Z-00-DX1,TCGA-C8-A132,IDC
463,TCGA-GM-A3XL-01Z-00-DX1,TCGA-GM-A3XL,IDC


**!!!Replace with your selected genes and predicted genes here!!!**

load the data file, we use the `tpm_unstranded` gene counts as features here, and select the higly 250 expressed for trianing and testing:

In [6]:
import warnings

# Filter or ignore specific warning categories
warnings.filterwarnings("ignore")
from glob import glob
from tqdm import tqdm

# load all bulk rna-seq data as a big dataframe
all_rna_file_path = glob("/data1/r20user3/shared_project/Hist2Cell/data/brca/*/*.rna_seq.augmented_star_gene_counts.tsv")
case_rna_df_list = []
for rna_file_path in tqdm(all_rna_file_path):
    case = rna_file_path.split("/")[-2]
    df = pd.read_csv(rna_file_path, sep="\t", skiprows=1, header=0)
    df = df[4:]
    brca_df = df[["gene_name", "tpm_unstranded"]]
    brca_df["gene_name"] = brca_df["gene_name"].str.split('.', expand=True)[0]
    brca_df = brca_df.drop_duplicates(subset=["gene_name"])
    new_brca_df = pd.DataFrame(columns=brca_df['gene_name'].values, data=[brca_df['tpm_unstranded'].values], index=[case])
    case_rna_df_list.append(new_brca_df)
    
all_case_rna_df = pd.concat(case_rna_df_list, join='inner')
all_case_rna_df

100%|██████████| 826/826 [08:18<00:00,  1.66it/s]


,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,CDR1,AL139190,AC119733,AC010980,AC007511,AL451106,AC073611,AC136977,AC078856,AL391628
TCGA-E2-A1IE-01Z-00-DX1,34.5847,1.1065,129.3534,14.4737,7.6050,3.9460,10.1275,46.4962,10.3253,84.8503,...,0.0,0.0202,0.0,0.0315,0.0,0.0,0.4421,0.0,0.0,0.0146
TCGA-AR-A1AT-01Z-00-DX1,28.3693,0.1866,92.0135,15.0553,8.4406,13.4883,28.8008,65.7654,10.5013,45.0618,...,0.0,0.0000,0.0,0.0928,0.0,0.0,0.0966,0.0,0.0,0.0286
TCGA-A2-A0CP-01Z-00-DX1,23.9018,0.8334,62.8190,5.5016,1.6630,6.9363,23.3521,42.0305,6.8102,27.3883,...,0.0,0.0000,0.0,0.3553,0.0,0.0,0.3237,0.0,0.0,0.0365
TCGA-XX-A89A-01Z-00-DX1,30.5954,53.6310,78.1094,11.0614,5.3765,19.6006,42.9337,50.8266,15.6718,34.5035,...,0.0,0.0000,0.0,0.3397,0.0,0.0,0.4898,0.0,0.0,0.0887
TCGA-AC-A7VC-01Z-00-DX1,67.1803,3.1455,89.0285,4.9424,6.3769,6.3647,40.8630,69.4266,6.9642,23.7224,...,0.0,0.2561,0.0,0.3414,0.0,0.0,0.1926,0.0,0.0,0.0088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-AN-A0XN-01Z-00-DX1,36.3313,0.1148,154.5065,8.5923,5.0383,12.4514,52.0587,65.2734,10.9230,57.6942,...,0.0,0.0367,0.0,0.1142,0.0,0.0,0.2230,0.0,0.0,0.0440
TCGA-AO-A0JD-01Z-00-DX1,60.3483,0.5546,180.0561,14.4989,18.7501,8.4510,6.5820,230.5040,21.3157,82.1567,...,0.0,0.0296,0.0,0.0690,0.0,0.0,0.0598,0.0,0.0,0.0071
TCGA-C8-A132-01Z-00-DX1,27.1198,0.3767,113.9529,14.7321,5.5065,11.3231,37.1017,59.7230,10.9275,56.1510,...,0.0,0.0603,0.0,0.1171,0.0,0.0,0.2195,0.0,0.0,0.0072
TCGA-GM-A3XL-01Z-00-DX1,19.3988,0.1661,178.5804,7.4212,12.0377,17.8720,5.1171,47.1452,10.8020,29.0449,...,0.0,0.0000,0.0,1.9827,0.0,0.0,0.2839,0.0,0.0,0.0459


In [10]:
# generate X_df accodring to index of y_df, and only keep the 250 highly expressed genes

X_all_df = all_case_rna_df.loc[y_df['Slide.ID'].values]
# Calculate the mean of each column
column_means = X_all_df.mean()
# Get the indices of the 250 columns with the largest means
largest_250_indices = column_means.nlargest(250).index
# Select only the 250 columns with the largest means
X_df = X_all_df[largest_250_indices]
X_df

,MT-CO3,MT-CO2,MT-CO1,MT-ND4,MT-RNR2,IGKC,MT-ATP6,MT-CYB,MT-ND1,MT-ND2,...,RHOA,MTND2P28,CCNI,IGHV1-18,DHCR24,EIF4G2,IGHG2,GPX4,RPL34,CAPNS1
TCGA-E2-A1IE-01Z-00-DX1,30900.1874,26822.6181,29375.1747,22667.7304,15009.7866,753.9009,14796.6468,14302.2374,12928.1309,12258.1969,...,294.9438,404.0426,453.8205,6.2595,905.7178,533.9820,16.0426,565.1618,379.6323,561.9105
TCGA-A2-A0CP-01Z-00-DX1,39306.9412,25094.3568,24000.6251,33217.3106,32516.4992,9276.1255,19015.1475,20216.3353,27286.8311,15230.4312,...,342.2215,390.8223,353.3905,96.4309,285.4083,263.1429,135.6249,584.4666,720.1975,535.7432
TCGA-C8-A278-01Z-00-DX1,20094.3687,22895.7568,17316.8824,23789.2826,17860.1077,98520.3064,9315.0443,9676.1987,14642.0299,14142.4577,...,248.7567,451.5729,177.5391,7278.3083,1396.3526,309.8565,2458.8750,261.4605,182.5878,234.5144
TCGA-EW-A3E8-01Z-00-DX1,20187.2624,21232.7971,17369.9080,10199.7578,8137.0962,25260.4345,10880.1943,8158.8972,7857.5138,8995.6317,...,310.6150,305.9994,246.7299,297.1930,555.9913,294.7764,693.3833,891.3653,505.1264,443.8636
TCGA-A2-A0YL-01Z-00-DX1,16301.6756,10369.7887,19770.3864,11909.6557,10370.6855,6846.1484,7959.6641,8663.0269,8836.5798,6193.1804,...,358.1975,205.1792,721.7781,449.1347,330.4806,602.1199,257.9670,416.2490,849.6970,403.2736
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-AC-A2FO-01Z-00-DX1,19282.3585,14330.5921,18322.3464,13374.5022,11956.7331,23388.8904,9716.0686,7852.1455,9484.5456,8341.9745,...,424.3607,238.5566,452.2834,340.8487,747.8411,368.6310,435.3714,475.0314,477.9048,488.0592
TCGA-E9-A1R0-01Z-00-DX1,18694.5741,17501.0767,20598.9313,16370.0644,17024.0218,19823.9750,11663.1076,10624.6989,11307.5982,10097.3565,...,474.7302,281.0403,719.4516,101.4068,412.8522,446.2653,1432.5050,588.1292,541.5835,481.8030
TCGA-C8-A132-01Z-00-DX1,17415.9456,15974.9115,14379.2635,16230.2905,4877.5044,25721.4950,8125.7196,9904.9217,7258.0574,8005.5425,...,672.3523,242.7185,396.3304,468.7033,228.8455,435.8329,1071.3002,297.5880,348.0956,485.3812
TCGA-GM-A3XL-01Z-00-DX1,12886.9021,20864.2807,10465.0858,11336.0069,4982.2771,32621.3538,5153.3512,7171.6260,7856.0675,10438.5180,...,304.1563,261.3059,292.8924,269.8561,434.1703,255.6212,667.5632,752.3878,727.9072,389.6083


merge X_df and y_df to get dataset_df and convert subtype label into categorical int values:

In [15]:
y_df = y_df.set_index('Slide.ID')
y_df[['Final Pathology']]

,Final Pathology
Slide.ID,
TCGA-E2-A1IE-01Z-00-DX1,IDC
TCGA-A2-A0CP-01Z-00-DX1,IDC
TCGA-C8-A278-01Z-00-DX1,IDC
TCGA-EW-A3E8-01Z-00-DX1,ILC
TCGA-A2-A0YL-01Z-00-DX1,ILC
...,...
TCGA-AC-A2FO-01Z-00-DX1,ILC
TCGA-E9-A1R0-01Z-00-DX1,IDC
TCGA-C8-A132-01Z-00-DX1,IDC


In [23]:
dataset_df = X_df.merge(y_df[['Final Pathology']], left_index=True, right_index=True, how='outer')
dataset_df.head(2)

,MT-CO3,MT-CO2,MT-CO1,MT-ND4,MT-RNR2,IGKC,MT-ATP6,MT-CYB,MT-ND1,MT-ND2,...,MTND2P28,CCNI,IGHV1-18,DHCR24,EIF4G2,IGHG2,GPX4,RPL34,CAPNS1,Final Pathology
TCGA-E2-A1IE-01Z-00-DX1,30900.1874,26822.6181,29375.1747,22667.7304,15009.7866,753.9009,14796.6468,14302.2374,12928.1309,12258.1969,...,404.0426,453.8205,6.2595,905.7178,533.9820,16.0426,565.1618,379.6323,561.9105,IDC
TCGA-A2-A0CP-01Z-00-DX1,39306.9412,25094.3568,24000.6251,33217.3106,32516.4992,9276.1255,19015.1475,20216.3353,27286.8311,15230.4312,...,390.8223,353.3905,96.4309,285.4083,263.1429,135.6249,584.4666,720.1975,535.7432,IDC


In [24]:
dataset_df['subtype'] = dataset_df['Final Pathology'].map({'IDC': 1,'ILC': 0})
dataset_df.drop('Final Pathology', axis=1, inplace=True)
dataset_df.head(2)

,MT-CO3,MT-CO2,MT-CO1,MT-ND4,MT-RNR2,IGKC,MT-ATP6,MT-CYB,MT-ND1,MT-ND2,...,MTND2P28,CCNI,IGHV1-18,DHCR24,EIF4G2,IGHG2,GPX4,RPL34,CAPNS1,subtype
TCGA-E2-A1IE-01Z-00-DX1,30900.1874,26822.6181,29375.1747,22667.7304,15009.7866,753.9009,14796.6468,14302.2374,12928.1309,12258.1969,...,404.0426,453.8205,6.2595,905.7178,533.9820,16.0426,565.1618,379.6323,561.9105,1
TCGA-A2-A0CP-01Z-00-DX1,39306.9412,25094.3568,24000.6251,33217.3106,32516.4992,9276.1255,19015.1475,20216.3353,27286.8311,15230.4312,...,390.8223,353.3905,96.4309,285.4083,263.1429,135.6249,584.4666,720.1975,535.7432,1


split dataset into 70% training and 30% testing, and fit a logistic regression model:

In [33]:
# split the initial dataset
from sklearn.model_selection import train_test_split

x = dataset_df.iloc[:,:-1]
y = dataset_df['subtype']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.7, random_state=2000)

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# create regressor
logreg = LogisticRegression(C=1e8, solver='newton-cg')
# fit the model with trianing data
logreg.fit(x_train, y_train)
# predictions of training data
y_pred_train = logreg.predict(x_train)
# predictions of test data
y_pred_test = logreg.predict(x_test)

# Calculate the accuracy of the predictions
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print("Training accuracy:", train_accuracy)
print("Test accuracy:", test_accuracy)

Training accuracy: 1.0
Test accuracy: 0.8128834355828221


In [35]:
from sklearn.metrics import roc_auc_score

# Predict probabilities of the test data
y_pred_prob_test = logreg.predict_proba(x_test)[:, 1]

# Calculate the AUC
test_auc = roc_auc_score(y_test, y_pred_prob_test)

print("Test AUC:", test_auc)

Test AUC: 0.8385780885780887


The performance seems to be comparable as STNet.